# 4.4. Modelo dimensional y capa de consumo

## Objetivo

Construir una capa analítica que permita consultar
los indicadores históricos, territoriales, tecnológicos
y predictivos del proyecto OSIPTEL.

## Diseño dimensional

Se utiliza una constelación de hechos con dimensiones
compartidas, manteniendo la granularidad y las unidades
de medida de cada fuente.

Las cantidades de conexiones representan existencias
al cierre de un periodo y no deben acumularse entre
trimestres.

Los porcentajes del INEI no deben sumarse entre ámbitos
geográficos ni asignarse directamente a departamentos.

Las predicciones se mantienen separadas de los valores
observados y conservan la identificación del modelo
que las generó.

In [2]:
from pathlib import Path
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROYECTO = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "run_pipeline.py").is_file()
    and (p / "notebooks").is_dir()
)

BASE = PROYECTO / "data" / "paquete_20261125_c5cc727"

RESULTADOS = PROYECTO / "results" / "modelo_dimensional"

RESULTADOS.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("OSIPTEL_Modelo_Dimensional")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

f1_nacional = spark.read.parquet(
    (BASE / "f1_historico_nacional").as_uri()
)

f1_territorial = spark.read.parquet(
    (BASE / "f1_historico_territorial").as_uri()
)

f4 = spark.read.parquet(
    (BASE / "f4_hogares_verificados").as_uri()
)

ARCHIVO_PREDICCIONES = (
    PROYECTO / "results" / "modelo_definitivo"
    / "prediccion_siguiente_trimestre.csv"
)

assert ARCHIVO_PREDICCIONES.is_file(), (
    "Falta generar el CSV del modelo definitivo"
)

predicciones = (
    spark.read
    .option("header", True)
    .csv(ARCHIVO_PREDICCIONES.as_uri())
)

print("Fuentes analiticas cargadas correctamente")

Fuentes analiticas cargadas correctamente


In [3]:
fact_conexiones_nacional = f1_nacional.select(
    "periodo",
    "conexiones_fijas",
    "filas_unicas",
    "filas_sin_departamento",
    "origen_sha256",
    "regla_version"
)

fact_conexiones_territorial = (
    f1_territorial
    .filter(
        F.col("periodo").between("2016-03", "2026-03")
    )
    .select(
        "periodo",
        "departamento",
        "operador",
        "tecnologia",
        "segmento",
        "rango_velocidad",
        "conexiones_fijas",
        "filas_origen",
        "filas_observadas",
        "origen_sha256",
        "regla_version"
    )
)

fact_hogares_internet = (
    f4.withColumn(
        "periodo_cierre",
        F.concat_ws(
            "-",
            F.substring("periodo", 1, 4),
            F.lpad(
                (
                    F.substring("periodo", 7, 1)
                    .cast("int") * 3
                ).cast("string"),
                2,
                "0"
            )
        )
    )
    .select(
        F.col("periodo").alias("trimestre_inei"),
        F.col("periodo_cierre").alias("periodo"),
        "ambito",
        "indicador",
        "porcentaje",
        "unidad",
        "pdf",
        "pagina_pdf",
        "sha256_pdf"
    )
)

ID_MODELO = "RF_A_operativo_c5cc727_v1"

fact_predicciones = predicciones.select(
    "departamento",
    F.col("periodo").alias("periodo_origen"),
    "periodo_prediccion",
    F.col("conexiones").cast("long").alias(
        "conexiones_observadas"
    ),
    F.col("prediccion").cast("double").alias(
        "conexiones_predichas"
    ),
    F.col("variacion_estimada").cast("double"),
    F.col("variacion_estimada_pct").cast("double"),
    F.lit(ID_MODELO).alias("id_modelo"),
    F.lit("c5cc727").alias("version_datos")
)

print("Cuatro tablas de hechos construidas")

Cuatro tablas de hechos construidas


In [4]:
dim_periodo = (
    fact_conexiones_nacional.select("periodo")

    .unionByName(
        fact_conexiones_territorial.select("periodo")
    )

    .unionByName(
        fact_hogares_internet.select("periodo")
    )

    .unionByName(
        fact_predicciones.select(
            F.col("periodo_origen").alias("periodo")
        )
    )

    .unionByName(
        fact_predicciones.select(
            F.col("periodo_prediccion").alias("periodo")
        )
    )

    .distinct()

    .withColumn(
        "fecha",
        F.last_day(
            F.to_date(
                F.concat(F.col("periodo"), F.lit("-01"))
            )
        )
    )

    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("trimestre", F.quarter("fecha"))

    .withColumn(
        "anio_trimestre",
        F.concat(
            F.col("anio"),
            F.lit("-T"),
            F.col("trimestre")
        )
    )

    .orderBy("periodo")
)

dim_territorio = (
    fact_conexiones_territorial
    .select("departamento")
    .distinct()
)

dim_operador = (
    fact_conexiones_territorial
    .select("operador")
    .distinct()
)

dim_tecnologia = (
    fact_conexiones_territorial
    .select("tecnologia")
    .distinct()
)

dim_segmento = (
    fact_conexiones_territorial
    .select("segmento")
    .distinct()
)

dim_velocidad = (
    fact_conexiones_territorial
    .select("rango_velocidad")
    .distinct()
)

dim_ambito = (
    fact_hogares_internet
    .select("ambito")
    .distinct()
)

dim_modelo = spark.createDataFrame(
    [(
        ID_MODELO,
        "Random Forest Regressor",
        50,
        3,
        42,
        "2026-03",
        "Operativo",
        "c5cc727"
    )],
    [
        "id_modelo",
        "algoritmo",
        "num_arboles",
        "profundidad_maxima",
        "semilla",
        "fin_entrenamiento",
        "tipo_modelo",
        "version_datos"
    ]
)

print("Ocho dimensiones construidas")

Ocho dimensiones construidas


In [5]:
tablas = {
    "fact_conexiones_nacional": fact_conexiones_nacional,
    "fact_conexiones_territorial": fact_conexiones_territorial,
    "fact_hogares_internet": fact_hogares_internet,
    "fact_predicciones": fact_predicciones,

    "dim_periodo": dim_periodo,
    "dim_territorio": dim_territorio,
    "dim_operador": dim_operador,
    "dim_tecnologia": dim_tecnologia,
    "dim_segmento": dim_segmento,
    "dim_velocidad": dim_velocidad,
    "dim_ambito": dim_ambito,
    "dim_modelo": dim_modelo
}

claves = {
    "fact_conexiones_nacional": ["periodo"],

    "fact_conexiones_territorial": [
        "periodo", "departamento", "operador",
        "tecnologia", "segmento", "rango_velocidad"
    ],

    "fact_hogares_internet": [
        "periodo", "ambito", "indicador"
    ],

    "fact_predicciones": [
        "departamento", "periodo_origen",
        "periodo_prediccion", "id_modelo"
    ],

    "dim_periodo": ["periodo"],
    "dim_territorio": ["departamento"],
    "dim_operador": ["operador"],
    "dim_tecnologia": ["tecnologia"],
    "dim_segmento": ["segmento"],
    "dim_velocidad": ["rango_velocidad"],
    "dim_ambito": ["ambito"],
    "dim_modelo": ["id_modelo"]
}

inventario = []

for nombre, df in tablas.items():

    filas = df.count()

    claves_unicas = (
        df.select(*claves[nombre])
        .distinct()
        .count()
    )

    assert filas == claves_unicas, (
        f"Claves duplicadas en {nombre}"
    )

    inventario.append({
        "tabla": nombre,
        "tipo": (
            "Hechos" if nombre.startswith("fact_")
            else "Dimension"
        ),
        "filas": filas,
        "columnas": len(df.columns),
        "claves_duplicadas": filas - claves_unicas
    })

relaciones = [
    ("fact_conexiones_nacional", "periodo",
     "dim_periodo", "periodo"),

    ("fact_conexiones_territorial", "periodo",
     "dim_periodo", "periodo"),

    ("fact_conexiones_territorial", "departamento",
     "dim_territorio", "departamento"),

    ("fact_conexiones_territorial", "operador",
     "dim_operador", "operador"),

    ("fact_conexiones_territorial", "tecnologia",
     "dim_tecnologia", "tecnologia"),

    ("fact_conexiones_territorial", "segmento",
     "dim_segmento", "segmento"),

    ("fact_conexiones_territorial", "rango_velocidad",
     "dim_velocidad", "rango_velocidad"),

    ("fact_hogares_internet", "periodo",
     "dim_periodo", "periodo"),

    ("fact_hogares_internet", "ambito",
     "dim_ambito", "ambito"),

    ("fact_predicciones", "departamento",
     "dim_territorio", "departamento"),

    ("fact_predicciones", "periodo_origen",
     "dim_periodo", "periodo"),

    ("fact_predicciones", "periodo_prediccion",
     "dim_periodo", "periodo"),

    ("fact_predicciones", "id_modelo",
     "dim_modelo", "id_modelo")
]

for hecho, fk, dimension, pk in relaciones:

    h = tablas[hecho]
    d = tablas[dimension]

    huerfanos = (
        h.join(d, h[fk] == d[pk], "left_anti")
        .count()
    )

    assert huerfanos == 0, (
        f"Relacion invalida: {hecho} -> {dimension}"
    )

for nombre, df in tablas.items():

    df.write.mode("overwrite").parquet(
        (RESULTADOS / nombre).as_uri()
    )

resumen_dimensional = pd.DataFrame(inventario)

display(resumen_dimensional)

resumen_dimensional.to_csv(
    RESULTADOS / "inventario_dimensional.csv",
    index=False
)

print("Relaciones validadas:", len(relaciones))
print("Tablas almacenadas:", len(tablas))

,tabla,tipo,filas,columnas,claves_duplicadas
0,fact_conexiones_nacional,Hechos,71,6,0
1,fact_conexiones_territorial,Hechos,126446,11,0
2,fact_hogares_internet,Hechos,20,9,0
3,fact_predicciones,Hechos,25,9,0
4,dim_periodo,Dimension,72,6,0
5,dim_territorio,Dimension,25,1,0
6,dim_operador,Dimension,151,1,0
7,dim_tecnologia,Dimension,7,1,0
8,dim_segmento,Dimension,2,1,0
9,dim_velocidad,Dimension,14,1,0


Relaciones validadas: 13
Tablas almacenadas: 12


## Resultados del modelo dimensional

Se implementó una constelación de hechos conformada por cuatro tablas de hechos y ocho dimensiones, almacenadas en formato Parquet mediante Apache Spark.

La tabla de hechos territoriales contiene 126.446 registros correspondientes al periodo comprendido entre marzo de 2016 y marzo de 2026. La tabla de predicciones almacena 25 estimaciones generadas por el modelo Random Forest.

Se validaron las claves de las 12 tablas y las 13 relaciones dimensionales, sin encontrar duplicidades ni registros huérfanos en las relaciones evaluadas.

El diseño permite consultar los indicadores históricos, tecnológicos, territoriales y predictivos, manteniendo separadas las cantidades observadas, las estimaciones del modelo y los porcentajes de acceso a internet del INEI.

Esta estructura constituye la base de la capa de consumo que posteriormente alimentará el dashboard analítico.